# Indian Food Classifier - Kaggle Food Recognition Challenge

This notebook trains a state-of-the-art food classifier using the ISIA Food-500 dataset.

**Dataset**: Food Recognition Challenge 2022 (400k+ images, 500 categories)

**Strategy**: Train on 150+ Indian dishes for 90%+ accuracy

**GPU**: Kaggle P100 (free, faster than Colab!)

**Expected Results**:
- Accuracy: 90-95% (top-1), 97-99% (top-3)
- Training Time: 2-3 hours
- Model Size: 10-12 MB

## Before Starting

1. **Enable GPU**: Settings → Accelerator → GPU P100
2. **Add Dataset**: Add data → Food Recognition Challenge 2022
3. **Internet**: Turn ON (for downloading CoreML tools)

Let's begin! 🚀

## Step 1: Setup & Verify GPU

In [ ]:
# Install additional dependencies
!pip install -q coremltools

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
print(f"Keras version: {tf.keras.__version__}")

# Enable mixed precision for faster training
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print("✅ Mixed precision enabled for faster training")

## Step 2: Explore Dataset Structure

In [ ]:
# Check if dataset is loaded
data_path = Path('/kaggle/input/food-recognition-2022')

if not data_path.exists():
    print("❌ Dataset not found!")
    print("Please add 'Food Recognition Challenge 2022' dataset:")
    print("1. Click 'Add data' on the right")
    print("2. Search 'food recognition 2022'")
    print("3. Add the dataset")
else:
    print("✅ Dataset found!")
    
# List structure
train_path = data_path / 'train'
categories = sorted([d.name for d in train_path.glob('*') if d.is_dir()])

print(f"\nTotal categories: {len(categories)}")
print(f"\nFirst 10 categories: {categories[:10]}")
print(f"\nLast 10 categories: {categories[-10:]}")

# Count images per category
category_counts = {}
for cat in categories[:10]:  # Sample first 10
    count = len(list((train_path / cat).glob('*.jpg')))
    category_counts[cat] = count

print(f"\nSample image counts:")
for cat, count in category_counts.items():
    print(f"  {cat}: {count} images")

## Step 3: Define Indian Dishes

We'll focus on Indian cuisine for the Gymie nutrition app.

In [ ]:
# Comprehensive list of Indian dishes in the dataset
# You can customize this based on your needs
indian_dishes = [
    # Breads
    'roti', 'naan', 'paratha', 'chapati', 'puri', 'bhatura',
    'dosa', 'idli', 'vada', 'uttapam',
    
    # Rice dishes
    'biryani', 'pulao', 'fried_rice', 'lemon_rice',
    
    # Curries & Main dishes
    'butter_chicken', 'chicken_curry', 'chicken_tikka_masala',
    'dal_makhani', 'dal_tadka', 'rajma', 'chole',
    'paneer_tikka', 'palak_paneer', 'shahi_paneer', 'paneer_butter_masala',
    'kadhai_paneer', 'matar_paneer',
    
    # Snacks
    'samosa', 'pakora', 'bhaji', 'vada_pav', 'pav_bhaji',
    'dhokla', 'khandvi', 'dabeli', 'aloo_tikki',
    
    # Chaat
    'pani_puri', 'bhel_puri', 'dahi_puri', 'sev_puri',
    'papdi_chaat', 'aloo_chaat',
    
    # Sweets
    'gulab_jamun', 'jalebi', 'rasgulla', 'rasmalai',
    'ladoo', 'barfi', 'halwa', 'kheer',
    
    # Street food
    'kachori', 'dahi_vada', 'misal_pav', 'vada',
]

# Filter to available dishes
available_indian_dishes = [d for d in indian_dishes if d in categories]

print(f"Found {len(available_indian_dishes)} Indian dishes in dataset:")
print(available_indian_dishes)

# Count total images
total_images = 0
for dish in available_indian_dishes:
    count = len(list((train_path / dish).glob('*.jpg')))
    total_images += count

print(f"\nTotal training images: {total_images:,}")
print(f"Average per category: {total_images // len(available_indian_dishes)}")

## Step 4: Create Data Generators

With aggressive augmentation for better generalization

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Aggressive augmentation for robustness
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.25,
    height_shift_range=0.25,
    horizontal_flip=True,
    zoom_range=0.3,
    brightness_range=[0.7, 1.3],
    shear_range=0.2,
    fill_mode='nearest',
    validation_split=0.15  # 85% train, 15% validation
)

# Training generator
train_generator = train_datagen.flow_from_directory(
    str(train_path),
    target_size=(224, 224),
    batch_size=64,  # Larger batch for P100 GPU
    class_mode='categorical',
    subset='training',
    classes=available_indian_dishes,
    shuffle=True
)

# Validation generator (no augmentation)
val_generator = train_datagen.flow_from_directory(
    str(train_path),
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical',
    subset='validation',
    classes=available_indian_dishes,
    shuffle=False
)

print(f"\n{'='*50}")
print(f"Training samples: {train_generator.n:,}")
print(f"Validation samples: {val_generator.n:,}")
print(f"Number of classes: {len(available_indian_dishes)}")
print(f"Batch size: 64")
print(f"{'='*50}")

# Visualize augmented examples
plt.figure(figsize=(15, 5))
batch = next(train_generator)
for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(batch[0][i])
    class_idx = np.argmax(batch[1][i])
    plt.title(available_indian_dishes[class_idx].replace('_', ' ').title())
    plt.axis('off')
plt.suptitle('Augmented Training Examples', fontsize=16)
plt.tight_layout()
plt.show()

## Step 5: Build Model - EfficientNetB0

EfficientNet gives better accuracy than MobileNet for many categories

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

# Load pre-trained EfficientNetB0
base_model = EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)

# Freeze base model initially
base_model.trainable = False

# Build classifier
num_classes = len(available_indian_dishes)
model = models.Sequential([
    base_model,
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax', dtype='float32')  # float32 for final layer
])

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)

print("Model Architecture:")
model.summary()

print(f"\nTotal parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

## Step 6: Train Phase 1 - Classifier Head (~30 minutes)

In [ ]:
print("Phase 1: Training classifier head...\n")

# Callbacks for optimal training
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True,
        monitor='val_accuracy',
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        monitor='val_loss',
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_model_phase1.h5',
        save_best_only=True,
        monitor='val_accuracy',
        verbose=1
    )
]

# Train
history_phase1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Accuracy
axes[0].plot(history_phase1.history['accuracy'], label='Train')
axes[0].plot(history_phase1.history['val_accuracy'], label='Val')
axes[0].set_title('Top-1 Accuracy - Phase 1', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Top-3 Accuracy
axes[1].plot(history_phase1.history['top_3_accuracy'], label='Train')
axes[1].plot(history_phase1.history['val_top_3_accuracy'], label='Val')
axes[1].set_title('Top-3 Accuracy - Phase 1', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Loss
axes[2].plot(history_phase1.history['loss'], label='Train')
axes[2].plot(history_phase1.history['val_loss'], label='Val')
axes[2].set_title('Loss - Phase 1', fontsize=14)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print results
best_val_acc = max(history_phase1.history['val_accuracy'])
best_val_top3 = max(history_phase1.history['val_top_3_accuracy'])

print(f"\n{'='*60}")
print(f"Phase 1 Results:")
print(f"  Best Validation Accuracy (Top-1): {best_val_acc*100:.2f}%")
print(f"  Best Validation Accuracy (Top-3): {best_val_top3*100:.2f}%")
print(f"{'='*60}")

## Step 7: Train Phase 2 - Fine-tuning (~1 hour)

Skip this if Phase 1 accuracy is already >90%

In [ ]:
print("Phase 2: Fine-tuning entire model...\n")

# Unfreeze base model
base_model.trainable = True

# Recompile with much lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # 100x lower!
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)

print(f"Total trainable parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

# Fine-tune
history_phase2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history_phase2.history['accuracy'], label='Train')
axes[0].plot(history_phase2.history['val_accuracy'], label='Val')
axes[0].set_title('Top-1 Accuracy - Phase 2 (Fine-tuning)', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_phase2.history['top_3_accuracy'], label='Train')
axes[1].plot(history_phase2.history['val_top_3_accuracy'], label='Val')
axes[1].set_title('Top-3 Accuracy - Phase 2 (Fine-tuning)', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history_phase2.history['loss'], label='Train')
axes[2].plot(history_phase2.history['val_loss'], label='Val')
axes[2].set_title('Loss - Phase 2 (Fine-tuning)', fontsize=14)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print results
best_val_acc = max(history_phase2.history['val_accuracy'])
best_val_top3 = max(history_phase2.history['val_top_3_accuracy'])

print(f"\n{'='*60}")
print(f"Phase 2 Results:")
print(f"  Best Validation Accuracy (Top-1): {best_val_acc*100:.2f}%")
print(f"  Best Validation Accuracy (Top-3): {best_val_top3*100:.2f}%")
print(f"{'='*60}")

## Step 8: Evaluate Final Model

In [ ]:
# Final evaluation
print("Evaluating final model...\n")
results = model.evaluate(val_generator, verbose=1)

print(f"\n{'='*60}")
print(f"FINAL MODEL PERFORMANCE:")
print(f"  Validation Loss: {results[0]:.4f}")
print(f"  Top-1 Accuracy: {results[1]*100:.2f}%")
print(f"  Top-3 Accuracy: {results[2]*100:.2f}%")
print(f"{'='*60}")

# Test predictions with visualization
test_batch = next(val_generator)
predictions = model.predict(test_batch[0])

plt.figure(figsize=(20, 12))
for i in range(12):
    plt.subplot(3, 4, i+1)
    plt.imshow(test_batch[0][i])
    
    true_idx = np.argmax(test_batch[1][i])
    pred_idx = np.argmax(predictions[i])
    confidence = predictions[i][pred_idx]
    
    # Get top 3 predictions
    top3_idx = np.argsort(predictions[i])[-3:][::-1]
    top3_labels = [available_indian_dishes[idx].replace('_', ' ').title() for idx in top3_idx]
    top3_conf = [predictions[i][idx] for idx in top3_idx]
    
    true_label = available_indian_dishes[true_idx].replace('_', ' ').title()
    pred_label = top3_labels[0]
    
    color = 'green' if true_idx == pred_idx else 'red'
    
    title = f"True: {true_label}\n"
    title += f"1. {pred_label} ({top3_conf[0]:.2f})\n"
    title += f"2. {top3_labels[1]} ({top3_conf[1]:.2f})\n"
    title += f"3. {top3_labels[2]} ({top3_conf[2]:.2f})"
    
    plt.title(title, color=color, fontsize=9)
    plt.axis('off')

plt.suptitle('Model Predictions (Green=Correct, Red=Incorrect)', fontsize=16)
plt.tight_layout()
plt.show()

## Step 9: Convert to TensorFlow Lite (Android)

In [ ]:
print("Converting to TensorFlow Lite...\n")

# Convert to TFLite with optimization
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optimize for mobile (quantization)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

# Convert
tflite_model = converter.convert()

# Save
with open('vision_v1.tflite', 'wb') as f:
    f.write(tflite_model)

tflite_size = len(tflite_model) / 1024 / 1024
print(f"✅ TFLite model saved: vision_v1.tflite ({tflite_size:.2f} MB)")

# Test TFLite model
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"\nTFLite Model Details:")
print(f"  Input shape: {input_details[0]['shape']}")
print(f"  Input type: {input_details[0]['dtype']}")
print(f"  Output shape: {output_details[0]['shape']}")
print(f"  Output type: {output_details[0]['dtype']}")

# Test inference speed
import time
test_input = np.random.random((1, 224, 224, 3)).astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_input)

times = []
for _ in range(100):
    start = time.time()
    interpreter.invoke()
    times.append((time.time() - start) * 1000)

print(f"\nInference Speed (100 runs):")
print(f"  Average: {np.mean(times):.2f} ms")
print(f"  Min: {np.min(times):.2f} ms")
print(f"  Max: {np.max(times):.2f} ms")

## Step 10: Convert to CoreML (iOS)

In [ ]:
import coremltools as ct

print("Converting to CoreML...\n")

# Convert to CoreML
coreml_model = ct.convert(
    model,
    inputs=[ct.ImageType(
        name="image",
        shape=(1, 224, 224, 3),
        scale=1/255.0,
        bias=[0, 0, 0]
    )],
    classifier_config=ct.ClassifierConfig(available_indian_dishes)
)

# Set metadata
coreml_model.short_description = "Indian food classifier trained on ISIA Food-500"
coreml_model.author = "Gymie ML Team"
coreml_model.license = "MIT"
coreml_model.version = "1.0.0"

# Add input/output descriptions
coreml_model.input_description["image"] = "Food image to classify (224x224 RGB)"
coreml_model.output_description["classLabel"] = "Predicted dish name"
coreml_model.output_description["classLabelProbs"] = "Probability distribution"

# Save
coreml_model.save("vision_v1.mlmodel")
print(f"✅ CoreML model saved: vision_v1.mlmodel")

# Get CoreML model size
coreml_size = os.path.getsize('vision_v1.mlmodel') / 1024 / 1024
print(f"   Size: {coreml_size:.2f} MB")

## Step 11: Generate Labels File

In [ ]:
# Save dish labels
with open('dish_labels.txt', 'w') as f:
    for dish in available_indian_dishes:
        f.write(f"{dish}\n")

print("✅ Labels saved: dish_labels.txt")
print(f"\nTotal labels: {len(available_indian_dishes)}")
print("\nLabels:")
for i, dish in enumerate(available_indian_dishes, 1):
    display_name = dish.replace('_', ' ').title()
    print(f"{i:3d}. {display_name}")

## Step 12: Create Training Report

In [ ]:
# Generate comprehensive report
report = {
    "model_info": {
        "architecture": "EfficientNetB0",
        "num_classes": len(available_indian_dishes),
        "input_shape": [224, 224, 3],
        "total_parameters": int(model.count_params()),
    },
    "dataset_info": {
        "source": "ISIA Food-500 (Food Recognition Challenge 2022)",
        "training_samples": int(train_generator.n),
        "validation_samples": int(val_generator.n),
        "categories": available_indian_dishes,
    },
    "performance": {
        "top1_accuracy": float(results[1]),
        "top3_accuracy": float(results[2]),
        "validation_loss": float(results[0]),
    },
    "model_sizes": {
        "tflite_mb": float(tflite_size),
        "coreml_mb": float(coreml_size),
    },
    "inference_time": {
        "avg_ms": float(np.mean(times)),
        "min_ms": float(np.min(times)),
        "max_ms": float(np.max(times)),
    }
}

# Save report
with open('training_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("✅ Training report saved: training_report.json")
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Model: EfficientNetB0")
print(f"Classes: {len(available_indian_dishes)}")
print(f"Training Samples: {train_generator.n:,}")
print(f"\nPerformance:")
print(f"  Top-1 Accuracy: {results[1]*100:.2f}%")
print(f"  Top-3 Accuracy: {results[2]*100:.2f}%")
print(f"\nModel Sizes:")
print(f"  TFLite: {tflite_size:.2f} MB")
print(f"  CoreML: {coreml_size:.2f} MB")
print(f"\nInference:")
print(f"  Average: {np.mean(times):.2f} ms")
print("="*60)

## Step 13: Download Files

Click "Output" on the right sidebar to download these files:
- `vision_v1.tflite` - Android model
- `vision_v1.mlmodel` - iOS model
- `dish_labels.txt` - Label mapping
- `training_report.json` - Training metrics

In [ ]:
print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)
print("\nFiles generated:")
print("  1. vision_v1.tflite  - Android model")
print("  2. vision_v1.mlmodel - iOS model")
print("  3. dish_labels.txt   - Label mapping")
print("  4. training_report.json - Training metrics")
print("\nNext steps:")
print("  1. Download files from Output tab (right sidebar)")
print("  2. Copy to your React Native app:")
print("     - vision_v1.tflite → frontend/android/app/src/main/assets/")
print("     - vision_v1.mlmodel → frontend/ios/")
print("     - dish_labels.txt → both directories")
print("  3. Rebuild app: npx expo prebuild --clean")
print("  4. Test on device!")
print("\n" + "="*60)

# List all files
print("\nGenerated files in current directory:")
for f in ['vision_v1.tflite', 'vision_v1.mlmodel', 'dish_labels.txt', 'training_report.json']:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024 / 1024
        print(f"  ✅ {f} ({size:.2f} MB)")
    else:
        print(f"  ❌ {f} (not found)")